
# Semantic Memory - Exploratory Data Analysis

**Paper context:** *The Role of System 1 and System 2 Semantic Memory Structure in Human and LLM Biases* (Abramski, Rossetti & Stella, 2026)

## 1. Introduction

This notebook explores the datasets used in the paper, focusing on semantic associations produced by humans and large language models (LLMs). The main goal is to understand how the data is structured, identify common patterns, and compare human and LLM-generated word associations.

We will use descriptive statistics and visualizations to explore the data, rather than reproduce the full experiments from the paper.



### 1.1 Environment Setup

We will use Python and Pandas to load, inspect, and analyze the datasets. NumPy will be used for numerical operations, while Plotly will help us create interactive visualizations.


In [1]:

import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)



## 2. Dataset Description

This analysis focuses on two sources of semantic word associations:

- **SWOW (Small World of Words):** A dataset containing word associations collected from human participants. Given a cue word, participants provide words that come to mind.

- **LWOW (Large World of Words):** A dataset of word associations generated by large language models. It allows us to explore how LLM-generated associations compare with human responses.

We will mainly focus on these two datasets to investigate differences and similarities in word association patterns.

The exact dataset structure, available columns, and number of observations will be examined after loading the data.



## 3. Data Loading & Initial Inspection

In this section, we load the SWOW and LWOW datasets and examine their structure.

We will check the available columns, data types, number of observations, and missing values before starting the analysis.


In [2]:
from pathlib import Path

DATA_DIR = Path("data")

print("Available datasets:")

for path in sorted(DATA_DIR.rglob("*.csv")):
    print(path)

Available datasets:
data/lwow/FA_Llama3.csv
data/lwow/FA_Mistral.csv
data/swow/SWOW-EN.R100.20180827.csv



### 3.1 Loading the SWOW Dataset

We will start by loading the SWOW dataset, which contains word associations collected from human participants.

After loading the data, we will inspect the first few rows to understand its structure and check whether the CSV was read correctly.


In [3]:
swow_path = DATA_DIR / "swow" / "SWOW-EN.R100.20180827.csv"

swow = pd.read_csv(
    swow_path,
    low_memory=False,
    keep_default_na=False,
    na_values=[""]
)

display(swow.head())


,Unnamed: 0,id,participantID,age,gender,nativeLanguage,country,education,created_at,cue,R1,R2,R3
0,1,29,3,33,Fe,United States,Australia,NA,2011-08-12 02:19:38,although,nevertheless,yet,but
1,2,30,3,33,Fe,United States,Australia,NA,2011-08-12 02:19:38,deal,no,cards,shake
2,3,31,3,33,Fe,United States,Australia,NA,2011-08-12 02:19:38,music,notes,band,rhythm
3,4,32,3,33,Fe,United States,Australia,NA,2011-08-12 02:19:38,inform,tell,rat on,NA
4,5,33,3,33,Fe,United States,Australia,NA,2011-08-12 02:19:38,way,path,via,method


In [4]:

print(f"Number of rows: {swow.shape[0]:,}")
print(f"Number of columns: {swow.shape[1]}")
print(f"Columns: {swow.columns.tolist()}")


Number of rows: 1,228,200
Number of columns: 13
Columns: ['Unnamed: 0', 'id', 'participantID', 'age', 'gender', 'nativeLanguage', 'country', 'education', 'created_at', 'cue', 'R1', 'R2', 'R3']



### 3.2 Data Types & Missing Values

Before preprocessing the data, we need to check the data types and identify missing values.

This will help us understand which columns might require cleaning and whether missing responses could affect our analysis.


In [5]:

swow.info()

<class 'pandas.DataFrame'>
RangeIndex: 1228200 entries, 0 to 1228199
Data columns (total 13 columns):
 #   Column          Non-Null Count    Dtype
---  ------          --------------    -----
 0   Unnamed: 0      1228200 non-null  int64
 1   id              1228200 non-null  int64
 2   participantID   1228200 non-null  int64
 3   age             1228200 non-null  int64
 4   gender          1228200 non-null  str  
 5   nativeLanguage  1228200 non-null  str  
 6   country         1223518 non-null  str  
 7   education       1228200 non-null  str  
 8   created_at      1228200 non-null  str  
 9   cue             1228200 non-null  str  
 10  R1              1228200 non-null  str  
 11  R2              1228200 non-null  str  
 12  R3              1228200 non-null  str  
dtypes: int64(4), str(9)
memory usage: 121.8 MB


In [6]:

missing_values = pd.DataFrame({
    "Missing Count": swow.isna().sum(),
    "Missing (%)": swow.isna().mean() * 100
})

display(missing_values.round(2))


,Missing Count,Missing (%)
Unnamed: 0,0,0.00
id,0,0.00
participantID,0,0.00
age,0,0.00
gender,0,0.00
nativeLanguage,0,0.00
country,4682,0.38
education,0,0.00
created_at,0,0.00
cue,0,0.00



**Initial observations:**

- Most columns have very few missing values according to the initial Pandas inspection.
- No missing cue words were detected after correcting the CSV parsing configuration.
- The response columns initially show no missing values, but this does not necessarily mean that every response is valid.
- Some missing responses may be represented by text markers such as `NA`, which require further investigation.

**Preprocessing considerations:**

We should avoid removing entire rows just because some responses are missing. An observation with valid R1 and R2 responses can still be useful even if R3 is unavailable.

We will investigate missing-value markers during data preparation before deciding how to handle them.



### 3.3 Loading the LWOW Datasets

We will now examine the word associations generated by Llama3 and Mistral.

First, we will load both datasets and inspect their structure. This will help us understand how LLM-generated associations are stored and whether they can be compared with the human responses from SWOW.


In [7]:

llama_path = DATA_DIR / "lwow" / "FA_Llama3.csv"
mistral_path = DATA_DIR / "lwow" / "FA_Mistral.csv"

llama = pd.read_csv(
    llama_path,
    low_memory=False,
    keep_default_na=False,
    na_values=[""]
)

mistral = pd.read_csv(
    mistral_path,
    low_memory=False,
    keep_default_na=False,
    na_values=[""]
)

print("Llama3:")
display(llama.head())

print("Mistral:")
display(mistral.head())


Llama3:


,cue,R1,R2,R3
0,a,absorbent,acoustic,accelerate
1,a,absorbent,acoustic,accelerate
2,a,absorbent,acoustic,accelerate
3,a,abundance,accentuate,affect
4,a,acorn,airplane,astronaut


Mistral:


,cue,R1,R2,R3
0,a,NaN,NaN,NaN
1,a,NaN,NaN,NaN
2,a,NaN,NaN,NaN
3,a,NaN,NaN,NaN
4,a,NaN,NaN,NaN


In [8]:

for name, df in [("Llama3", llama), ("Mistral", mistral)]:
    print(f"\n{name}")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print(f"Column names: {df.columns.tolist()}")



Llama3
Rows: 1,154,500
Columns: 4
Column names: ['cue', 'R1', 'R2', 'R3']

Mistral
Rows: 1,154,500
Columns: 4
Column names: ['cue', 'R1', 'R2', 'R3']


In [9]:

for name, df in [("Llama3", llama), ("Mistral", mistral)]:
    print(f"\n{name}")

    print("Missing values (%):")
    print((df[["R1", "R2", "R3"]].isna().mean() * 100).round(2))

    all_missing = df[["R1", "R2", "R3"]].isna().all(axis=1).sum()

    print(f"Rows with all responses missing: {all_missing:,}")
    print(f"Percentage: {all_missing / len(df) * 100:.2f}%")



Llama3
Missing values (%):
R1    1.90
R2    1.93
R3    6.17
dtype: float64
Rows with all responses missing: 21,917
Percentage: 1.90%

Mistral
Missing values (%):
R1    4.06
R2    4.87
R3    7.98
dtype: float64
Rows with all responses missing: 46,924
Percentage: 4.06%



**Initial observations:**

- Both LWOW datasets contain 1,154,500 rows and share the same four columns: `cue`, `R1`, `R2`, and `R3`.
- Mistral has more missing responses than Llama3 across all three response positions (this is why we see NaN when returning `head()`)
- R3 has the highest percentage of missing values in both datasets.
- Around 1.90% of Llama3 rows and 4.06% of Mistral rows have no recorded responses.
- A small inspection of the original Mistral CSV confirmed that some response fields are genuinely empty. These missing values are not simply caused by Pandas interpreting text as NaN


**Preprocessing considerations:**

Both datasets appear suitable for further analysis. However, differences in missing responses could affect comparisons of association frequency and lexical diversity.

We should account for the number of valid responses when comparing models and investigate how missing values are represented in the original files.



### 3.4 Dataset Structure Comparison

Now that we have inspected SWOW, Llama3, and Mistral separately, we can compare their basic structure.

We will check the number of observations, available columns, and unique cue words in each dataset.

We will also investigate how many cue words are shared across all three datasets, since a common set of cues will be important for our later comparative analysis.


In [10]:

datasets = {
    "SWOW": swow,
    "Llama3": llama,
    "Mistral": mistral
}

dataset_summary = []

for name, df in datasets.items():
    dataset_summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Unique cues": df["cue"].nunique(),
        "Missing cues": df["cue"].isna().sum()
    })

summary_df = pd.DataFrame(dataset_summary)

display(summary_df)


,Dataset,Rows,Columns,Unique cues,Missing cues
0,SWOW,1228200,13,12282,0
1,Llama3,1154500,4,11545,0
2,Mistral,1154500,4,11545,0


In [11]:

swow_cues = set(swow["cue"].dropna().unique())
llama_cues = set(llama["cue"].dropna().unique())
mistral_cues = set(mistral["cue"].dropna().unique())

common_cues = swow_cues & llama_cues & mistral_cues

print(f"SWOW unique cues: {len(swow_cues):,}")
print(f"Llama3 unique cues: {len(llama_cues):,}")
print(f"Mistral unique cues: {len(mistral_cues):,}")
print(f"Common cues: {len(common_cues):,}")


SWOW unique cues: 12,282
Llama3 unique cues: 11,545
Mistral unique cues: 11,545
Common cues: 11,104



**Common Cue Words ~ Explanation**

To compare human and LLM-generated word associations, we need to identify the cue words that appear in all three datasets.

We use Python sets and the intersection operator (`&`) to find the cue words shared across all three datasets.

The intersection contains only cue words that are present in SWOW, Llama3, and Mistral.

**Initial observations:**

- SWOW contains **12,282** unique cue words.
- Llama3 and Mistral each contain **11,545** unique cue words.
- **11,104 cue words** are shared across all three datasets.
- This gives us a large common vocabulary for comparing human and LLM-generated associations.

Using the same cue words helps us make fairer comparisons between humans and LLMs. Otherwise, differences in association patterns could be influenced by the words included in each dataset rather than by the responses themselves.

However, a shared cue word does not guarantee that valid responses are available in all three datasets. We will check this during data preparation.



### 3.5 Summary of Initial Findings

After loading and inspecting SWOW, Llama3, and Mistral, we can summarize the main findings.

**Dataset structure:**

- SWOW contains 1,228,200 rows and 13 columns, including participant information and word associations.
- Llama3 and Mistral each contain 1,154,500 rows and 4 columns (`cue`, `R1`, `R2`, `R3`).
- All three datasets represent associations using a cue word and up to three responses.

**Missing values:**

- No missing cue words were detected after correcting the CSV parsing configuration.
- Mistral has more missing responses than Llama3.
- R3 has the highest percentage of missing responses in both LWOW datasets.
- SWOW may contain additional missing responses represented by text markers, which will be investigated during preprocessing.

**Common vocabulary:**

- SWOW contains 12,282 unique cue words.
- Llama3 and Mistral each contain 11,545 unique cue words.
- A total of 11,104 cue words are shared across all three datasets.

The shared vocabulary provides a useful starting point for comparing human and LLM-generated associations.

**Next steps:**

We will investigate duplicates, missing-value markers, and text formatting inconsistencies before transforming the data into a common representation for EDA.



## 4. Data Preparation

Before performing descriptive statistics and exploratory analysis, we need to check the quality and consistency of our datasets.

In this section, we will identify potential issues, clean the data where necessary, and prepare a common format for comparing human and LLM-generated word associations.

The original datasets will remain unchanged.



### 4.1 Data Quality Checks

We will start by checking for duplicate rows and possible inconsistencies in the word association data.

Since SWOW contains participant information while LWOW contains generated responses, we need to be careful when interpreting duplicates. Repeated associations may be valid observations rather than data errors.


In [12]:

duplicate_summary = []

for name, df in datasets.items():
    duplicates = df.duplicated().sum()

    duplicate_summary.append({
        "Dataset": name,
        "Total Rows": len(df),
        "Duplicate Rows": duplicates,
        "Duplicate (%)": round(duplicates / len(df) * 100, 2)
    })

display(pd.DataFrame(duplicate_summary))


,Dataset,Total Rows,Duplicate Rows,Duplicate (%)
0,SWOW,1228200,0,0.00
1,Llama3,1154500,812451,70.37
2,Mistral,1154500,407749,35.32



**Initial observations:**

- SWOW has no fully duplicated rows, but this is expected since it contains unique identifiers and participant information.
- Llama3 has a high percentage of duplicate rows (70.37%), compared to Mistral (35.32%).
- In LWOW, repeated rows may represent identical associations generated across multiple runs, rather than data errors.
- These repetitions could be useful for studying how frequently certain associations are produced.

**Preprocessing considerations:**

We should not automatically remove duplicate rows, since doing so would change the observed frequency of word associations. The difference in duplicate percentages is interesting, but we cannot conclude that one model has lower semantic diversity based on this metric alone. We will keep the repeated observations and investigate response diversity later in the analysis.


In [13]:
# question, are they technically or semantically duplicated ?
association_columns = ["cue", "R1", "R2", "R3"]

for name, df in datasets.items():
    print(f"\n{name}")

    for col in association_columns:
        empty_strings = df[col].eq("").sum()
        whitespace_only = df[col].str.fullmatch(r"\s+").sum()

        print(
            f"{col}: "
            f"{empty_strings:,} empty strings, "
            f"{whitespace_only:,} whitespace-only values"
        )



SWOW
cue: 0 empty strings, 0 whitespace-only values
R1: 0 empty strings, 0 whitespace-only values
R2: 0 empty strings, 0 whitespace-only values
R3: 0 empty strings, 0 whitespace-only values

Llama3
cue: 0 empty strings, 0 whitespace-only values
R1: 0 empty strings, 0 whitespace-only values
R2: 0 empty strings, 0 whitespace-only values
R3: 0 empty strings, 0 whitespace-only values

Mistral
cue: 0 empty strings, 0 whitespace-only values
R1: 0 empty strings, 3 whitespace-only values
R2: 0 empty strings, 1 whitespace-only values
R3: 0 empty strings, 0 whitespace-only values


In [14]:

response_cols = ["R1", "R2", "R3"]

for name, df in [("Llama3", llama), ("Mistral", mistral)]:
    valid_rows = df[df[response_cols].notna().any(axis=1)]

    duplicates = valid_rows.duplicated().sum()

    print(f"\n{name}")
    print(f"Rows with at least one response: {len(valid_rows):,}")
    print(f"Duplicate rows: {duplicates:,}")
    print(f"Duplicate percentage: {duplicates / len(valid_rows) * 100:.2f}%")



Llama3
Rows with at least one response: 1,132,583
Duplicate rows: 790,796
Duplicate percentage: 69.82%

Mistral
Rows with at least one response: 1,107,576
Duplicate rows: 369,741
Duplicate percentage: 33.38%



**Duplicate analysis - Additional observations:**

After excluding rows where all three responses are missing, the duplicate percentages remain high:

- Llama3: 69.82% duplicate rows.
- Mistral: 33.38% duplicate rows.

This suggests that most duplicates are not caused by completely empty response rows.

Llama3 repeats identical response combinations more frequently than Mistral in the available data.

However, these percentages measure repeated combinations of R1, R2, and R3 for the same cue, not semantic diversity directly.

**Decision:**

We will keep duplicate observations because their frequency may be relevant to our analysis. We will investigate response-level diversity separately during descriptive statistics.


In [15]:
# [dog] or [ dog] or [Dog ]
for name, df in datasets.items():
    print(f"\n{name}")

    for col in association_columns:
        values = df[col].dropna()

        extra_spaces = values.ne(values.str.strip()).sum()
        uppercase = values.ne(values.str.lower()).sum()

        print(
            f"{col}: "
            f"{extra_spaces:,} values with surrounding whitespace, "
            f"{uppercase:,} values with uppercase characters"
        )



SWOW
cue: 0 values with surrounding whitespace, 39,400 values with uppercase characters
R1: 0 values with surrounding whitespace, 79,744 values with uppercase characters
R2: 0 values with surrounding whitespace, 126,228 values with uppercase characters
R3: 0 values with surrounding whitespace, 210,983 values with uppercase characters

Llama3
cue: 0 values with surrounding whitespace, 0 values with uppercase characters
R1: 7,606 values with surrounding whitespace, 0 values with uppercase characters
R2: 11,341 values with surrounding whitespace, 0 values with uppercase characters
R3: 17,292 values with surrounding whitespace, 0 values with uppercase characters

Mistral
cue: 0 values with surrounding whitespace, 0 values with uppercase characters
R1: 4 values with surrounding whitespace, 0 values with uppercase characters
R2: 6 values with surrounding whitespace, 0 values with uppercase characters
R3: 11 values with surrounding whitespace, 0 values with uppercase characters



### 4.2 Data Cleaning

Based on the previous data quality checks, we identified differences in text formatting and missing values across the datasets.

Our goal is to prepare consistent word association data while preserving useful information, especially repeated associations.

We will keep the original DataFrames unchanged and apply cleaning operations to separate copies.


In [16]:

cleaned_datasets = {
    name: df.copy()
    for name, df in datasets.items()
}

for name, df in cleaned_datasets.items():
    print(f"{name}: {len(df):,} rows")


SWOW: 1,228,200 rows
Llama3: 1,154,500 rows
Mistral: 1,154,500 rows


In [17]:

# treat SWOW's original "NA" markers as missing responses

swow_clean = cleaned_datasets["SWOW"]
response_cols = ["R1", "R2", "R3"]

for col in response_cols:
    swow_clean[col] = swow_clean[col].replace("NA", pd.NA)

print("SWOW missing responses after cleaning:")
print(swow_clean[response_cols].isna().sum())


SWOW missing responses after cleaning:
R1     31044
R2     79713
R3    170053
dtype: int64


In [18]:
# whitespace and capitalization cleaning

for name, df in cleaned_datasets.items():
    for col in association_columns:
        df[col] = df[col].str.strip().str.lower()

    print(f"{name}: text normalization completed")


SWOW: text normalization completed
Llama3: text normalization completed
Mistral: text normalization completed


In [19]:

# check cleaned datasets for whitespace and capitalization issues

for name, df in cleaned_datasets.items():
    original = datasets[name]

    changed_values = sum(
        (
            original[col].notna()
            & df[col].notna()
            & original[col].ne(df[col]).fillna(False)
        ).sum()
        for col in association_columns
    )

    remaining_whitespace = sum(
        df[col].dropna().ne(df[col].dropna().str.strip()).sum()
        for col in association_columns
    )

    remaining_uppercase = sum(
        df[col].dropna().ne(df[col].dropna().str.lower()).sum()
        for col in association_columns
    )

    print(f"\n{name}")
    print(f"Original rows: {len(original):,}")
    print(f"Cleaned rows: {len(df):,}")
    print(f"Non-null values changed: {changed_values:,}")
    print(f"Remaining surrounding whitespace: {remaining_whitespace:,}")
    print(f"Remaining uppercase values: {remaining_uppercase:,}")



SWOW
Original rows: 1,228,200
Cleaned rows: 1,228,200
Non-null values changed: 175,545
Remaining surrounding whitespace: 0
Remaining uppercase values: 0

Llama3
Original rows: 1,154,500
Cleaned rows: 1,154,500
Non-null values changed: 36,239
Remaining surrounding whitespace: 0
Remaining uppercase values: 0

Mistral
Original rows: 1,154,500
Cleaned rows: 1,154,500
Non-null values changed: 21
Remaining surrounding whitespace: 0
Remaining uppercase values: 0



**Text normalization : Results:**

- SWOW: 175,545 non-null values were modified, mainly due to differences in capitalization.
- Llama3: 36,239 values were modified by removing surrounding whitespace.
- Mistral: only 21 values required normalization.
- All three datasets now have no remaining uppercase characters or surrounding whitespace in their association columns.
- The number of observations remained unchanged.

The original datasets were preserved, and the normalization was applied only to separate copies.

This helps ensure that formatting differences do not artificially increase the number of unique words during our analysis.



**Missing response markers in SWOW:**

Further inspection showed that SWOW uses the text value `NA` for many missing responses.

- R1: 31,044 missing response markers (2.53%).
- R2: 79,713 missing response markers (6.49%).
- R3: 170,053 missing response markers (13.85%).

These values closely match the missing-value percentages observed during our initial inspection.

We decided to treat `NA` as a missing response marker in SWOW, while preserving valid words such as `null` and `none`.

This allows us to exclude unavailable responses from later calculations without losing meaningful semantic associations.


## Note

**CSV parsing issue:**

During data cleaning, we discovered that 100 SWOW observations had been incorrectly identified as having missing cue words.

After inspecting the original CSV, we confirmed that all 100 observations contained the valid cue word `null`.

Pandas had automatically interpreted this word as a missing value.

To prevent this, we updated the CSV loading configuration to preserve textual values such as `null`, while still treating empty fields as missing.

This is important because incorrectly removing these values could affect word frequencies and the structure of the semantic network.


In [20]:

# check possible missing-value markers in SWOW

response_cols = ["R1", "R2", "R3"]

markers = ["NA", "N/A", "null", "none", ""]

for col in response_cols:
    print(f"\n{col}")

    counts = swow[col].value_counts(dropna=False)

    for marker in markers:
        print(f"{marker!r}: {counts.get(marker, 0):,}")



R1
'NA': 31,044
'N/A': 0
'null': 40
'none': 218
'': 0

R2
'NA': 79,713
'N/A': 0
'null': 27
'none': 175
'': 0

R3
'NA': 170,053
'N/A': 0
'null': 24
'none': 183
'': 0


In [21]:

# count NA responses by position

for col in response_cols:
    count = swow[col].eq("NA").sum()
    percentage = count / len(swow) * 100

    print(f"{col}: {count:,} NA values ({percentage:.2f}%)")


R1: 31,044 NA values (2.53%)
R2: 79,713 NA values (6.49%)
R3: 170,053 NA values (13.85%)


In [22]:

# final validation of cleaned datasets

for name, df in cleaned_datasets.items():
    original = datasets[name]

    print(f"\n{name}")
    print(f"Rows preserved: {len(df) == len(original)}")
    print(f"Missing cues: {df['cue'].isna().sum():,}")
    print(f"Unique cues: {df['cue'].nunique():,}")
    print(f"Missing responses: {df[response_cols].isna().sum().to_dict()}")

print("\nSWOW 'null' cue occurrences:")
print(swow_clean["cue"].eq("null").sum())



SWOW
Rows preserved: True
Missing cues: 0
Unique cues: 12,282
Missing responses: {'R1': 31044, 'R2': 79713, 'R3': 170053}

Llama3
Rows preserved: True
Missing cues: 0
Unique cues: 11,545
Missing responses: {'R1': 21917, 'R2': 22264, 'R3': 71270}

Mistral
Rows preserved: True
Missing cues: 0
Unique cues: 11,545
Missing responses: {'R1': 46924, 'R2': 56272, 'R3': 92098}

SWOW 'null' cue occurrences:
100



### 4.3 Data Transformation

The datasets currently store word associations in a wide format, with up to three responses (R1, R2, R3) for each cue word.

To make the data easier to analyze, we will convert it into a long format, where each cue-response association becomes a separate row.

This representation will also help us construct semantic networks, where words are represented as nodes and associations as directed edges.


In [23]:

# transform SWOW from wide to long format

swow_long = swow_clean.melt(
    id_vars=["cue"],
    value_vars=["R1", "R2", "R3"],
    var_name="position",
    value_name="response"
)

# Remove associations without a valid response
swow_long = swow_long.dropna(subset=["response"])

display(swow_long.head(10))


,cue,position,response
0,although,R1,nevertheless
1,deal,R1,no
2,music,R1,notes
3,inform,R1,tell
4,way,R1,path
5,none,R1,zilch
6,extra,R1,plus
7,will,R1,free
8,paper,R1,book
9,w,R1,v


In [24]:

# validation check that the long format preserves all valid associations

expected = swow_clean[response_cols].notna().sum().sum()
actual = len(swow_long)

print(f"Expected valid associations: {expected:,}")
print(f"Actual long-format rows: {actual:,}")
print(f"Counts match: {expected == actual}")

print("\nResponses by position:")
print(swow_long["position"].value_counts().sort_index())


Expected valid associations: 3,403,790
Actual long-format rows: 3,403,790
Counts match: True

Responses by position:
position
R1    1197156
R2    1148487
R3    1058147
Name: count, dtype: int64



**SWOW transformation : Validation:**

- The wide-to-long transformation produced 3,403,790 valid cue-response associations.
- The number of resulting rows matches the number of valid responses in the cleaned SWOW dataset.
- R1 contains the most responses (1,197,156), followed by R2 (1,148,487) and R3 (1,058,147).
- Repeated associations were preserved, since they may be useful for calculating association frequencies.

Each row in the long-format dataset now represents one observed cue-response association, which can later be represented as a directed edge in a semantic network.


In [25]:

# transform LWOW datasets from wide to long format

lwow_long = {}

for name in ["Llama3", "Mistral"]:
    df = cleaned_datasets[name]

    long_df = df.melt(
        id_vars=["cue"],
        value_vars=response_cols,
        var_name="position",
        value_name="response"
    )

    lwow_long[name] = long_df.dropna(subset=["response"])

    print(f"{name}: {len(lwow_long[name]):,} valid associations")
    display(lwow_long[name].head(5))


Llama3: 3,348,049 valid associations


,cue,position,response
0,a,R1,absorbent
1,a,R1,absorbent
2,a,R1,absorbent
3,a,R1,abundance
4,a,R1,acorn


Mistral: 3,268,206 valid associations


,cue,position,response
13,a,R1,airplane
14,a,R1,also
15,a,R1,an
16,a,R1,an
17,a,R1,an


In [26]:

# validate LWOW wide-to-long transformation

for name in ["Llama3", "Mistral"]:
    expected = cleaned_datasets[name][response_cols].notna().sum().sum()
    actual = len(lwow_long[name])

    print(f"\n{name}")
    print(f"Expected valid associations: {expected:,}")
    print(f"Actual long-format rows: {actual:,}")
    print(f"Counts match: {expected == actual}")

    print("\nResponses by position:")
    print(lwow_long[name]["position"].value_counts().sort_index())



Llama3
Expected valid associations: 3,348,049
Actual long-format rows: 3,348,049
Counts match: True

Responses by position:
position
R1    1132583
R2    1132236
R3    1083230
Name: count, dtype: int64

Mistral
Expected valid associations: 3,268,206
Actual long-format rows: 3,268,206
Counts match: True

Responses by position:
position
R1    1107576
R2    1098228
R3    1062402
Name: count, dtype: int64



#### 4.3.1 Building Weighted Edge Lists

After converting the datasets to long format, we can represent each cue-response association as a directed connection.

The cue word represents the source node, while the response represents the target node.

Since the same association may appear multiple times, we will group identical cue-response pairs and count their occurrences. This frequency will be used as the edge weight in our exploratory network representation.


In [27]:
# build a weighted edge list from SWOW associations

swow_edges = (
    swow_long
    .groupby(["cue", "response"])
    .size()
    .reset_index(name="weight")
    .rename(columns={
        "cue": "source",
        "response": "target"
    })
)

display(swow_edges.head(10))


,source,target,weight
0,a,2,1
1,a,a,1
2,a,a horse,1
3,a,aardvark,1
4,a,abc,2
5,a,accomplish,1
6,a,achieve,1
7,a,adorable,1
8,a,afghan,1
9,a,ah,1


In [28]:

# validate SWOW weighted edge list

total_associations = len(swow_long)
total_weight = swow_edges["weight"].sum()

print(f"Original associations: {total_associations:,}")
print(f"Sum of edge weights: {total_weight:,}")
print(f"Counts match: {total_associations == total_weight}")

print(f"\nUnique directed edges: {len(swow_edges):,}")
print(f"Unique source nodes: {swow_edges['source'].nunique():,}")
print(f"Unique target nodes: {swow_edges['target'].nunique():,}")


Original associations: 3,403,790
Sum of edge weights: 3,403,790
Counts match: True

Unique directed edges: 1,394,792
Unique source nodes: 12,282
Unique target nodes: 130,489


In [29]:

# build weighted edge lists for Llama3 and Mistral

lwow_edges = {}

for name, df in lwow_long.items():
    lwow_edges[name] = (
        df.groupby(["cue", "response"])
        .size()
        .reset_index(name="weight")
        .rename(columns={
            "cue": "source",
            "response": "target"
        })
    )

    print(f"\n{name}")
    print(f"Unique directed edges: {len(lwow_edges[name]):,}")
    display(lwow_edges[name].head(5))



Llama3
Unique directed edges: 795,231


,source,target,weight
0,a,aardvark,2
1,a,aardwolf,2
2,a,abacus,1
3,a,absorbent,3
4,a,abundance,1



Mistral
Unique directed edges: 443,144


,source,target,weight
0,a,aardvark,2
1,a,about,1
2,a,above,1
3,a,abundance,1
4,a,advice,1


In [30]:

# validate weighted edge lists and count network nodes
# V = V source OR V target

all_edges = {
    "SWOW": swow_edges,
    **lwow_edges
}

all_long = {
    "SWOW": swow_long,
    **lwow_long
}

network_summary = []

for name, edges in all_edges.items():
    associations = len(all_long[name])
    total_weight = edges["weight"].sum()

    source_nodes = set(edges["source"])
    target_nodes = set(edges["target"])
    all_nodes = source_nodes | target_nodes

    network_summary.append({
        "Dataset": name,
        "Associations": associations,
        "Unique edges": len(edges),
        "Unique nodes": len(all_nodes),
        "Weight sum": total_weight,
        "Valid": associations == total_weight
    })

network_summary_df = pd.DataFrame(network_summary)

display(network_summary_df)


,Dataset,Associations,Unique edges,Unique nodes,Weight sum,Valid
0,SWOW,3403790,1394792,130539,3403790,True
1,Llama3,3348049,795231,101689,3348049,True
2,Mistral,3268206,443144,42058,3268206,True



**Network transformation: Initial observations:**

- All three datasets were successfully transformed from raw cue-response associations into weighted directed edge lists.
- The sum of edge weights matches the number of valid associations in each dataset.
- SWOW contains 130,539 unique nodes and 1,394,792 unique directed edges.
- Llama3 contains 101,689 unique nodes and 795,231 unique directed edges.
- Mistral contains 42,058 unique nodes and 443,144 unique directed edges.
- Although the datasets contain a similar number of observed associations, their numbers of unique nodes and edges differ considerably.

These results describe the networks constructed for our exploratory analysis. Further comparisons should account for differences in dataset size, response availability, and shared cue words.

The edge weights represent observed association frequencies, rather than a direct measure of semantic similarity.



### 4.4 Feature Engineering

After transforming the datasets into a common format, we can calculate additional features to better understand word association patterns.

We will start by computing the total number of valid responses and the number of unique responses for each cue word.

These features will help us explore association frequencies and response diversity across humans and LLMs.


In [31]:

# calculate basic response features for each cue

cue_features = {}

for name, df in all_long.items():
    features = (
        df.groupby("cue")
        .agg(
            response_count=("response", "size"),
            unique_responses=("response", "nunique")
        )
        .reset_index()
    )

    cue_features[name] = features

    print(f"\n{name}")
    print(f"Cues with valid responses: {len(features):,}")
    display(features.head(5))



SWOW
Cues with valid responses: 12,282


,cue,response_count,unique_responses
0,a,270,117
1,a few,269,105
2,a little,268,85
3,a lot,280,103
4,aardvark,269,117



Llama3
Cues with valid responses: 11,359


,cue,response_count,unique_responses
0,a,300,133
1,a few,300,58
2,a little,300,42
3,a lot,300,92
4,aardvark,300,63



Mistral
Cues with valid responses: 11,392


,cue,response_count,unique_responses
0,a,232,85
1,a few,191,57
2,a little,285,55
3,a lot,238,57
4,aardvark,262,45



#### Association Diversity

To measure how varied the responses are for each cue word, we calculate the association diversity ratio.

The formula is:

$$
D(c) = \frac{U(c)}{N(c)}
$$

Where:

- $D(c)$ = association diversity for cue word $c$.
- $U(c)$ = number of unique responses for cue word $c$.
- $N(c)$ = total number of valid responses for cue word $c$.

The value ranges from 0 to 1:

- **Closer to 1:** a larger proportion of responses are unique.
- **Closer to 0:** more responses are repeated.

For example, if a cue has 100 valid responses and 40 unique responses:

$$
D(c) = \frac{40}{100} = 0.4
$$

**Limitation:**

This ratio is sensitive to the number of observed responses. Cue words with different response counts may have different diversity scores simply because of sample size.

We will consider this limitation when comparing human and LLM-generated associations.


In [32]:

# calculate association diversity for each cue

for name, df in cue_features.items():
    df["association_diversity"] = (
        df["unique_responses"] / df["response_count"]
    )

    print(f"\n{name}")
    display(df.head(5))



SWOW


,cue,response_count,unique_responses,association_diversity
0,a,270,117,0.433333
1,a few,269,105,0.390335
2,a little,268,85,0.317164
3,a lot,280,103,0.367857
4,aardvark,269,117,0.434944



Llama3


,cue,response_count,unique_responses,association_diversity
0,a,300,133,0.443333
1,a few,300,58,0.193333
2,a little,300,42,0.140000
3,a lot,300,92,0.306667
4,aardvark,300,63,0.210000



Mistral


,cue,response_count,unique_responses,association_diversity
0,a,232,85,0.366379
1,a few,191,57,0.298429
2,a little,285,55,0.192982
3,a lot,238,57,0.239496
4,aardvark,262,45,0.171756


In [33]:

# validate association diversity values

for name, df in cue_features.items():
    valid_range = df["association_diversity"].between(0, 1).all()
    valid_counts = (
        df["unique_responses"] <= df["response_count"]
    ).all()

    print(f"\n{name}")
    print(f"Diversity values within [0, 1]: {valid_range}")
    print(f"Unique responses <= total responses: {valid_counts}")
    print(f"Min diversity: {df['association_diversity'].min():.4f}")
    print(f"Max diversity: {df['association_diversity'].max():.4f}")



SWOW
Diversity values within [0, 1]: True
Unique responses <= total responses: True
Min diversity: 0.1638
Max diversity: 0.7166

Llama3
Diversity values within [0, 1]: True
Unique responses <= total responses: True
Min diversity: 0.0100
Max diversity: 1.0000

Mistral
Diversity values within [0, 1]: True
Unique responses <= total responses: True
Min diversity: 0.0200
Max diversity: 0.7451



#### Most Frequent Response

For each cue word, we want to identify how frequently its most common response appears.

This feature can help us understand whether the observed associations are concentrated around a particular response or spread across multiple responses.

We will use the weighted edge lists created earlier, where each edge weight represents the number of occurrences of a cue-response association.


In [34]:

# calculate the frequency of the most common response per cue

for name, edges in all_edges.items():
    top_frequency = (
        edges.groupby("source")["weight"]
        .max()
        .rename("top_response_frequency")
    )

    cue_features[name] = cue_features[name].merge(
        top_frequency,
        left_on="cue",
        right_index=True,
        how="left"
    )

    print(f"\n{name}")
    display(cue_features[name].head(5))



SWOW


,cue,response_count,unique_responses,association_diversity,top_response_frequency
0,a,270,117,0.433333,32
1,a few,269,105,0.390335,29
2,a little,268,85,0.317164,31
3,a lot,280,103,0.367857,61
4,aardvark,269,117,0.434944,50



Llama3


,cue,response_count,unique_responses,association_diversity,top_response_frequency
0,a,300,133,0.443333,13
1,a few,300,58,0.193333,22
2,a little,300,42,0.140000,23
3,a lot,300,92,0.306667,14
4,aardvark,300,63,0.210000,11



Mistral


,cue,response_count,unique_responses,association_diversity,top_response_frequency
0,a,232,85,0.366379,56
1,a few,191,57,0.298429,27
2,a little,285,55,0.192982,73
3,a lot,238,57,0.239496,58
4,aardvark,262,45,0.171756,44



#### Top Response Proportion

The most frequent response count shows how often the dominant association appears for a cue word.

However, different cue words may have different numbers of valid responses. To make this measure easier to compare, we calculate the proportion of responses represented by the most frequent association.

$$
P_{\text{top}}(c) = \frac{f_{\max}(c)}{N(c)}
$$

Where:

- $f_{\max}(c)$ = frequency of the most common response for cue $c$.
- $N(c)$ = total number of valid responses for cue $c$.

A higher value means that a larger proportion of responses is concentrated around one association.


In [35]:

# calculate the proportion of the most frequent response

for name, df in cue_features.items():
    df["top_response_proportion"] = (
        df["top_response_frequency"] / df["response_count"]
    )

    print(f"\n{name}")
    display(df.head(5))



SWOW


,cue,response_count,unique_responses,association_diversity,top_response_frequency,top_response_proportion
0,a,270,117,0.433333,32,0.118519
1,a few,269,105,0.390335,29,0.107807
2,a little,268,85,0.317164,31,0.115672
3,a lot,280,103,0.367857,61,0.217857
4,aardvark,269,117,0.434944,50,0.185874



Llama3


,cue,response_count,unique_responses,association_diversity,top_response_frequency,top_response_proportion
0,a,300,133,0.443333,13,0.043333
1,a few,300,58,0.193333,22,0.073333
2,a little,300,42,0.140000,23,0.076667
3,a lot,300,92,0.306667,14,0.046667
4,aardvark,300,63,0.210000,11,0.036667



Mistral


,cue,response_count,unique_responses,association_diversity,top_response_frequency,top_response_proportion
0,a,232,85,0.366379,56,0.241379
1,a few,191,57,0.298429,27,0.141361
2,a little,285,55,0.192982,73,0.256140
3,a lot,238,57,0.239496,58,0.243697
4,aardvark,262,45,0.171756,44,0.167939



#### Mean Response Length

We will also calculate the average length of responses for each cue word.

This feature describes a simple lexical property of the associations and can help us explore differences in the types of responses produced by humans and LLMs.

The length is measured in characters, including spaces in multi-word responses.

This metric does not measure semantic complexity or the quality of an association.


In [36]:

# calculate the mean response length for each cue

for name, df in all_long.items():
    mean_length = (
        df.assign(response_length=df["response"].str.len())
        .groupby("cue")["response_length"]
        .mean()
        .rename("mean_response_length")
    )

    cue_features[name] = cue_features[name].merge(
        mean_length,
        left_on="cue",
        right_index=True,
        how="left"
    )

    print(f"\n{name}")
    display(cue_features[name].head(5))



SWOW


,cue,response_count,unique_responses,association_diversity,top_response_frequency,top_response_proportion,mean_response_length
0,a,270,117,0.433333,32,0.118519,4.285185
1,a few,269,105,0.390335,29,0.107807,6.048327
2,a little,268,85,0.317164,31,0.115672,5.082090
3,a lot,280,103,0.367857,61,0.217857,5.582143
4,aardvark,269,117,0.434944,50,0.185874,6.204461



Llama3


,cue,response_count,unique_responses,association_diversity,top_response_frequency,top_response_proportion,mean_response_length
0,a,300,133,0.443333,13,0.043333,6.333333
1,a few,300,58,0.193333,22,0.073333,6.583333
2,a little,300,42,0.140000,23,0.076667,6.943333
3,a lot,300,92,0.306667,14,0.046667,8.006667
4,aardvark,300,63,0.210000,11,0.036667,10.616667



Mistral


,cue,response_count,unique_responses,association_diversity,top_response_frequency,top_response_proportion,mean_response_length
0,a,232,85,0.366379,56,0.241379,5.780172
1,a few,191,57,0.298429,27,0.141361,5.120419
2,a little,285,55,0.192982,73,0.256140,5.101754
3,a lot,238,57,0.239496,58,0.243697,5.113445
4,aardvark,262,45,0.171756,44,0.167939,6.793893


In [37]:

# Final validation of engineered features

for name, df in cue_features.items():
    valid = (
        df["response_count"].gt(0).all()
        and df["unique_responses"].le(df["response_count"]).all()
        and df["association_diversity"].between(0, 1).all()
        and df["top_response_frequency"].le(df["response_count"]).all()
        and df["top_response_proportion"].between(0, 1).all()
        and df["mean_response_length"].gt(0).all()
        and df.isna().sum().sum() == 0
    )

    print(f"{name}:")
    print(f"  Cues: {len(df):,}")
    print(f"  Features valid: {valid}")


SWOW:
  Cues: 12,282
  Features valid: True
Llama3:
  Cues: 11,359
  Features valid: True
Mistral:
  Cues: 11,392
  Features valid: True



### 4.5 Prepared Data Summary

In this section, we prepared the SWOW, Llama3, and Mistral datasets for exploratory analysis.

**Data cleaning:**

- Preserved the original datasets and performed cleaning on separate copies.
- Normalized text by removing surrounding whitespace and converting words to lowercase.
- Corrected CSV parsing to preserve valid words such as `null` and `none`.
- Treated the `NA` response markers in SWOW as missing values.
- Preserved repeated associations because their frequencies are relevant to the analysis.

**Data transformation:**

- Converted the datasets from wide format (R1, R2, R3) into long format.
- Built weighted directed edge lists by grouping identical cue-response pairs.
- Identified unique nodes and edges for each dataset.
- Verified that the sum of edge weights matches the number of valid associations.

**Feature engineering:**

For each cue word with at least one valid response, we calculated:

- `response_count`: Total number of valid responses.
- `unique_responses`: Number of distinct responses.
- `association_diversity`: Proportion of unique responses.
- `top_response_frequency`: Frequency of the most common response.
- `top_response_proportion`: Proportion represented by the most common response.
- `mean_response_length`: Average response length in characters.

All engineered features passed the consistency checks.

**Limitations:**

The prepared datasets may still differ in the number of available responses and cue words. Our diversity measures describe observed lexical variation, not semantic complexity or cognitive mechanisms.

We will account for these differences during the comparative analysis.



## 5. Descriptive Statistics

After preparing the datasets and computing cue-level features, we can explore their statistical properties.

We will examine response counts, lexical diversity, association frequencies, and response lengths to better understand the patterns found in human and LLM-generated associations.

These descriptive statistics will provide a starting point for the exploratory and comparative analyses.



### 5.1 Descriptive Overview

We will start by calculating summary statistics for the features created in the previous section.

For each dataset, we will examine the mean, median, standard deviation, minimum, and maximum values.

This will help us understand the typical values and variability of each feature before exploring their distributions.


In [38]:

feature_columns = [
    "response_count",
    "unique_responses",
    "association_diversity",
    "top_response_frequency",
    "top_response_proportion",
    "mean_response_length"
]

descriptive_stats = {}

for name, df in cue_features.items():
    stats = df[feature_columns].agg(
        ["mean", "median", "std", "min", "max"]
    ).T

    descriptive_stats[name] = stats

    print(f"\n{name}")
    display(stats.round(3))



SWOW


,mean,median,std,min,max
response_count,277.136,282.000,17.383,115.000,300.000
unique_responses,113.564,113.000,21.392,47.000,191.000
association_diversity,0.411,0.406,0.081,0.164,0.717
top_response_frequency,39.043,36.000,17.170,7.000,97.000
top_response_proportion,0.141,0.129,0.062,0.029,0.383
mean_response_length,6.085,6.044,0.714,3.483,9.281



Llama3


,mean,median,std,min,max
response_count,294.749,300.00,23.073,12.000,300.000
unique_responses,70.009,71.00,20.399,3.000,213.000
association_diversity,0.239,0.24,0.077,0.010,1.000
top_response_frequency,13.411,12.00,7.550,1.000,100.000
top_response_proportion,0.046,0.04,0.027,0.013,0.727
mean_response_length,7.663,7.36,1.798,3.393,25.340



Mistral


,mean,median,std,min,max
response_count,286.886,292.000,18.114,24.000,300.000
unique_responses,38.900,36.000,15.639,6.000,129.000
association_diversity,0.138,0.125,0.065,0.020,0.745
top_response_frequency,69.248,72.000,20.068,3.000,100.000
top_response_proportion,0.240,0.248,0.065,0.035,0.381
mean_response_length,6.505,6.356,1.251,2.853,13.850



**Initial observations:**

- SWOW has the highest average number of unique responses per cue (113.56), followed by Llama3 (70.01) and Mistral (38.90).
- The average association diversity ratio is also highest in SWOW (0.411), compared to Llama3 (0.239) and Mistral (0.138).
- Mistral has the highest average top response proportion (0.240), suggesting that its responses are more concentrated around a dominant association.
- Llama3 has the lowest average top response proportion (0.046), despite having fewer unique responses than SWOW.
- Llama3 produces the longest responses on average (7.66 characters).

**Interpretation:**

These results suggest differences in how responses are distributed across human and LLM-generated associations.

However, lexical diversity and dominant response concentration measure different properties. A dataset can contain fewer unique responses without having a strongly dominant response.

The results are descriptive and do not directly measure semantic diversity or cognitive mechanisms.

Further comparisons should account for differences in the number of valid responses and the cue words available in each dataset.


In [ ]:
#Compare mean association diversity across datasets

mean_diversity = pd.DataFrame([
    {
        "Dataset": name,
        "Mean Diversity": df["association_diversity"].mean()
    }
    for name, df in cue_features.items()
])

fig = px.bar(
    mean_diversity,
    x="Dataset",
    y="Mean Diversity",
    color="Dataset",
    text="Mean Diversity",
    title="Mean Association Diversity by Dataset",
    color_discrete_map={
        "SWOW": "#3B82F6",
        "Llama3": "#10B981",
        "Mistral": "#F59E0B"
    }
)

fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")

fig.update_layout(
    yaxis_title="Mean association diversity",
    yaxis_range=[0, 0.5],
    showlegend=False,
    template="plotly_white"
)

fig.show()
